# Phần 3 - Mô hình dự báo doanh thu (Sales Forecasting)

Notebook này xây dựng pipeline dự báo cột `Revenue` cho giai đoạn test 2023-01-01 đến 2024-07-01. Theo đúng mô tả Phần 3 trong PDF, mô hình sử dụng `sales.csv` làm nguồn train chính đến hết 2022-12-31. Vì bộ local không công bố `sales_test.csv`, notebook dùng `sample_submission.csv` làm template để lấy ngày test và giữ nguyên thứ tự dòng nộp.

## Tóm tắt phương pháp

1. **Chia dữ liệu đúng đề:** train là toàn bộ dòng `sales.csv` có ngày không vượt quá 2022-12-31. Test/template là giai đoạn 2023-01-01 đến 2024-07-01. Không tự xáo trộn, không random split.
2. **Chống leakage:** mọi feature tại ngày `t` chỉ dùng thông tin train/quá khứ trước ngày `t`. Notebook **có dùng Revenue/COGS của train** để học mô hình, nhưng **không dùng Revenue/COGS của test** làm feature. Cột `COGS` trong template chỉ được giữ lại ở file nộp để đúng schema, không đưa vào mô hình dự báo `Revenue`.
3. **Feature engineering từ train Revenue/COGS:** dùng đầy đủ `Revenue` và `COGS` của **tập train** trong `sales.csv` để tạo lag 1-730 ngày, rolling mean/median/std/min/max, EWM, momentum, year-over-year ratio, margin và các tương tác COGS-margin. Đây là hợp lệ và là lõi của supervised time-series forecasting.
4. **Mô hình:** dự báo `COGS` nội sinh trước, sau đó dùng `COGS` dự báo như tín hiệu quy mô để dự báo `Revenue`. Ensemble gồm LightGBM, XGBoost, HistGradientBoosting, Ridge và seasonal baselines. Các boosting model tối ưu absolute error/MAE.
5. **Tối ưu MAE:** rolling-origin CV đúng chiều thời gian tạo OOF predictions. Ensemble weights được tune trực tiếp theo MAE, có trọng số ưu tiên fold gần nhất vì test thật nằm ngay sau 2022-12-31.
6. **Hiệu chỉnh bias có kiểm soát:** correction theo tháng được học từ fold gần nhất, có shrink và clip để giảm bias mùa vụ nhưng tránh overfit quá mức.
7. **Explainability:** xuất feature importance tổng hợp từ tree models và Ridge, kèm diễn giải kinh doanh trong markdown/report.


## 1. Thiết lập môi trường và tham số

Notebook này là pipeline tái lập cho Phần 3. Toàn bộ đặc trưng dùng để huấn luyện được tạo từ `sales.csv` và trường ngày. `Revenue`/`COGS` của tập train được dùng hợp lệ để tạo lag, rolling, EWM, margin và seasonal signal; `Revenue`/`COGS` của test/template không được đưa vào feature.

Mục tiêu tối ưu là đa metric: MAE vẫn là trọng tâm vì Kaggle hiển thị MAE, nhưng phần chọn ensemble có thêm RMSE và R2 để tránh dự báo quá khớp MAE làm phân phối bị méo.


In [ ]:
"""Leakage-free sales forecasting pipeline for Datathon 2026 Round 1.

The script reads data/sales.csv and splits it by date: train is up to
2022-12-31, test is 2023-01-01 to 2024-07-01. If the local sales.csv only
contains the train period, data/sample_submission.csv is used as the test
template.
It deliberately does not use Revenue or COGS from the submission/test rows as
model features. The final submission keeps the exact Date order from
sample_submission.csv and writes model-predicted Revenue/COGS columns. COGS is
predicted internally from train history; test/template COGS is never used as a
feature.
"""

from __future__ import annotations

import argparse
import json
import math
import random
import warnings
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

try:
    from statsmodels.tsa.forecasting.theta import ThetaModel
    from statsmodels.tsa.holtwinters import ExponentialSmoothing
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    STATSMODELS_AVAILABLE = True
except Exception:
    STATSMODELS_AVAILABLE = False


warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

try:
    ROOT = Path(__file__).resolve().parents[1]
except NameError:
    ROOT = Path.cwd().resolve()
    if ROOT.name == "03_modelling":
        ROOT = ROOT.parent
DATA_DIR = ROOT / "data"
REPORT_DIR = ROOT / "04_report_data"
TRAIN_END = pd.Timestamp("2022-12-31")
TEST_START = pd.Timestamp("2023-01-01")
TEST_END = pd.Timestamp("2024-07-01")

FAST_MODE = False  # Đổi thành True nếu chỉ muốn smoke-test nhanh trước khi chạy final trên Colab.
USE_STATSMODELS_CANDIDATES = False  # Classical models được giữ dạng tùy chọn vì khá chậm và dễ overfit OOF.

LAGS = [1, 2, 3, 7, 14, 21, 28, 35, 56, 91, 120, 182, 273, 364, 365, 371, 546, 728, 730]
ROLL_WINDOWS = [7, 14, 21, 28, 56, 91, 120, 182, 273, 365]
EWM_SPANS = [7, 14, 28, 56, 91, 182]

# Ensemble objective: MAE là chính vì Kaggle hiện score bằng MAE, nhưng RMSE/R2
# vẫn tham gia để tránh nghiệm quá khớp MAE và tạo forecast méo phân phối.
ENSEMBLE_OBJECTIVE_WEIGHTS = {"MAE": 0.70, "RMSE": 0.20, "R2": 0.10}
ENSEMBLE_MAE_TOLERANCE = 0.012  # chỉ xét nghiệm có weighted MAE trong 1.2% so với MAE tốt nhất.

COGS_MODEL_COLUMNS = [
    "cogs_lgb",
    "cogs_xgb",
    "cogs_hgb",
    "cogs_ridge",
    "cogs_lgb_l2",
    "cogs_xgb_l2",
    "cogs_hgb_l2",
    "cogs_lgb_raw",
    "cogs_xgb_raw",
    "cogs_hgb_raw",
    "cogs_season364",
    "cogs_season365",
    "cogs_season371",
    "cogs_season730",
    "cogs_season_blend",
    "cogs_doy_window",
    "cogs_local_trend",
    "cogs_ts_ets",
    "cogs_ts_theta",
    "cogs_ts_sarimax_weekly",
]

REVENUE_MODEL_COLUMNS = [
    "rev_lgb",
    "rev_xgb",
    "rev_hgb",
    "rev_ridge",
    "rev_lgb_l2",
    "rev_xgb_l2",
    "rev_hgb_l2",
    "rev_ratio_lgb",
    "rev_lgb_raw",
    "rev_xgb_raw",
    "rev_hgb_raw",
    "rev_lgb_nocogs",
    "rev_xgb_nocogs",
    "rev_hgb_nocogs",
    "rev_ridge_nocogs",
    "rev_lgb_raw_nocogs",
    "rev_xgb_raw_nocogs",
    "rev_hgb_raw_nocogs",
    "rev_margin_roll_28",
    "rev_margin_roll_91",
    "rev_margin_roll_365",
    "rev_margin_month",
    "rev_season364",
    "rev_season365",
    "rev_season371",
    "rev_season730",
    "rev_season_blend",
    "rev_doy_window",
    "rev_local_trend",
    "rev_ts_ets",
    "rev_ts_theta",
    "rev_ts_sarimax_weekly",
]

# Initial weights are only a fallback. The CV cell overwrites them with
# MAE-tuned OOF weights before creating the final submission.
DEFAULT_COGS_WEIGHTS = np.ones(len(COGS_MODEL_COLUMNS), dtype=float)
DEFAULT_COGS_WEIGHTS = DEFAULT_COGS_WEIGHTS / DEFAULT_COGS_WEIGHTS.sum()

DEFAULT_REVENUE_WEIGHTS = np.ones(len(REVENUE_MODEL_COLUMNS), dtype=float)
DEFAULT_REVENUE_WEIGHTS = DEFAULT_REVENUE_WEIGHTS / DEFAULT_REVENUE_WEIGHTS.sum()

# Public test is immediately after 2022-12-31, so final tuning is recency-weighted.
# We do not use 100% recent fold because the report also needs a stable model,
# not only a leaderboard-fit model.
FINAL_TUNING_FOLD_WEIGHTS = {
    "long_2020_2021H1": 0.10,
    "long_2021_2022H1": 0.18,
    "calendar_2022": 0.27,
    "recent_2021H2_2022": 0.30,
    "short_recent_2022H2": 0.15,
}

# Learned from the newest validation fold after MAE-tuned blending.
# Shrink + clip keeps the correction conservative and reduces overfitting risk.
APPLY_MONTH_CORRECTION = True
MONTH_CORRECTION_CLIP = (0.93, 1.07)
MONTH_CORRECTION_SHRINK = 0.35
MONTH_CORRECTION_FACTORS = pd.Series(dtype=float)

STATSMODELS_TAIL_DAYS = 1460
SARIMAX_TAIL_DAYS = 1095

# Avoid OOF overfitting where one candidate dominates the ensemble.
DEFAULT_SINGLE_MODEL_CAP = 0.35
MODEL_WEIGHT_CAPS = {
    "rev_ts_theta": 0.08,
    "rev_ts_ets": 0.08,
    "rev_ts_sarimax_weekly": 0.08,
    "cogs_ts_theta": 0.10,
    "cogs_ts_ets": 0.10,
    "cogs_ts_sarimax_weekly": 0.10,
}


# Strict Part 3 setup: model features are derived from sales.csv and Date only.
# Revenue/COGS from TRAIN are used heavily for lag/rolling/margin features.
# sample_submission.csv is used only as the required test template when sales_test.csv is not available locally.


## 1.1. Cấu trúc lưu mô hình và hàm đánh giá

Cell này định nghĩa object lưu model đã fit, danh sách feature, median imputation và các metric theo đúng PDF: MAE, RMSE, R2. RMSE được tính bằng `sqrt(mean_squared_error)` để tương thích với nhiều phiên bản scikit-learn, tránh lỗi tham số `squared=False`.


In [ ]:
@dataclass
class FittedBundle:
    models: dict
    cogs_cols: list[str]
    revenue_cols: list[str]
    cogs_medians: pd.Series
    revenue_medians: pd.Series
    origin: pd.Timestamp


def _metric_dict(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "R2": float(r2_score(y_true, y_pred)),
    }



## 1.2. Đặc trưng lịch và mùa vụ

Cell này tạo các feature biết trước trong tương lai: vị trí ngày trong tuần/tháng/năm, cờ cuối tuần/cuối tháng/cuối quý và Fourier terms cho chu kỳ tuần/năm. Đây là nhóm feature an toàn vì chỉ phụ thuộc vào `Date`, không phụ thuộc vào doanh thu tương lai.


In [ ]:
def calendar_features(dates: pd.Series | list[pd.Timestamp], origin: pd.Timestamp) -> pd.DataFrame:
    dates = pd.to_datetime(pd.Series(dates))
    out = pd.DataFrame(index=dates.index)
    out["days_from_start"] = (dates - origin).dt.days.astype(float)
    out["year"] = dates.dt.year.astype(float)
    out["month"] = dates.dt.month.astype(float)
    out["day"] = dates.dt.day.astype(float)
    out["dow"] = dates.dt.dayofweek.astype(float)
    out["weekofyear"] = dates.dt.isocalendar().week.astype(float)
    out["dayofyear"] = dates.dt.dayofyear.astype(float)
    out["quarter"] = dates.dt.quarter.astype(float)
    out["is_weekend"] = (dates.dt.dayofweek >= 5).astype(float)
    out["is_month_start"] = dates.dt.is_month_start.astype(float)
    out["is_month_end"] = dates.dt.is_month_end.astype(float)
    out["is_quarter_start"] = dates.dt.is_quarter_start.astype(float)
    out["is_quarter_end"] = dates.dt.is_quarter_end.astype(float)
    out["days_in_month"] = dates.dt.days_in_month.astype(float)
    out["month_progress"] = (dates.dt.day - 1) / dates.dt.days_in_month

    for k in range(1, 7):
        out[f"sin_doy_{k}"] = np.sin(2 * np.pi * k * out["dayofyear"] / 365.25)
        out[f"cos_doy_{k}"] = np.cos(2 * np.pi * k * out["dayofyear"] / 365.25)

    for k in range(1, 4):
        out[f"sin_woy_{k}"] = np.sin(2 * np.pi * k * out["weekofyear"] / 52.18)
        out[f"cos_woy_{k}"] = np.cos(2 * np.pi * k * out["weekofyear"] / 52.18)

    out["sin_dow"] = np.sin(2 * np.pi * out["dow"] / 7)
    out["cos_dow"] = np.cos(2 * np.pi * out["dow"] / 7)
    out["sin_month"] = np.sin(2 * np.pi * out["month"] / 12)
    out["cos_month"] = np.cos(2 * np.pi * out["month"] / 12)


    out["days_to_month_end"] = (out["days_in_month"] - out["day"]).astype(float)
    out["is_payday_window"] = ((out["day"] <= 5) | (out["day"] >= 25)).astype(float)
    out["is_mid_month"] = ((out["day"] >= 10) & (out["day"] <= 20)).astype(float)

    for month in range(1, 13):
        out[f"month_{month}"] = (out["month"] == month).astype(float)
    for dow in range(7):
        out[f"dow_{dow}"] = (out["dow"] == dow).astype(float)

    return out


## 1.3. Đặc trưng lag, rolling và dự báo tuần tự

Vì bài toán dự báo xa 548 ngày, notebook tạo feature theo đúng cách một hệ thống production sẽ chạy: mỗi ngày test được dự báo tuần tự, sau đó forecast của ngày đó được đưa vào history cho các ngày sau. Nhờ vậy lag/rolling trong test không nhìn thấy nhãn thật tương lai.

Các nhóm feature chính gồm lag ngắn hạn, lag mùa vụ 364/365/371/730 ngày, rolling mean/median/std/min/max, EWM, momentum ratio/difference, margin Revenue/COGS lịch sử, và các baseline time-series như seasonal blend, same-day-of-year window và local trend.


In [ ]:
def make_training_features(
    frame: pd.DataFrame,
    *,
    include_current_cogs: bool,
    origin: pd.Timestamp,
) -> pd.DataFrame:
    frame = frame.sort_values("date").reset_index(drop=True)
    features = calendar_features(frame["date"], origin)

    for col in ["Revenue", "COGS"]:
        series = frame[col].astype(float)
        shifted = series.shift(1)
        prefix = col.lower()

        for lag in LAGS:
            features[f"{prefix}_lag_{lag}"] = series.shift(lag)

        for window in ROLL_WINDOWS:
            min_periods = max(3, window // 3)
            rolling = shifted.rolling(window, min_periods=min_periods)
            features[f"{prefix}_roll_mean_{window}"] = rolling.mean()
            features[f"{prefix}_roll_median_{window}"] = rolling.median()
            features[f"{prefix}_roll_std_{window}"] = rolling.std()
            features[f"{prefix}_roll_min_{window}"] = rolling.min()
            features[f"{prefix}_roll_max_{window}"] = rolling.max()

        for span in EWM_SPANS:
            features[f"{prefix}_ewm_{span}"] = shifted.ewm(
                span=span, adjust=False, min_periods=3
            ).mean()

        features[f"{prefix}_lag1_div_lag7"] = (
            features[f"{prefix}_lag_1"] / features[f"{prefix}_lag_7"].replace(0, np.nan)
        )
        features[f"{prefix}_lag7_div_lag28"] = (
            features[f"{prefix}_lag_7"] / features[f"{prefix}_lag_28"].replace(0, np.nan)
        )
        features[f"{prefix}_lag364_div_lag728"] = (
            features[f"{prefix}_lag_364"] / features[f"{prefix}_lag_728"].replace(0, np.nan)
        )
        features[f"{prefix}_roll7_div_roll28"] = (
            features[f"{prefix}_roll_mean_7"]
            / features[f"{prefix}_roll_mean_28"].replace(0, np.nan)
        )
        features[f"{prefix}_roll28_div_roll91"] = (
            features[f"{prefix}_roll_mean_28"]
            / features[f"{prefix}_roll_mean_91"].replace(0, np.nan)
        )
        features[f"{prefix}_roll91_div_roll365"] = (
            features[f"{prefix}_roll_mean_91"]
            / features[f"{prefix}_roll_mean_365"].replace(0, np.nan)
        )
        features[f"{prefix}_lag365_div_roll365"] = (
            features[f"{prefix}_lag_365"]
            / features[f"{prefix}_roll_mean_365"].replace(0, np.nan)
        )
        features[f"{prefix}_lag1_minus_lag7"] = features[f"{prefix}_lag_1"] - features[f"{prefix}_lag_7"]
        features[f"{prefix}_roll7_minus_roll28"] = features[f"{prefix}_roll_mean_7"] - features[f"{prefix}_roll_mean_28"]
        features[f"{prefix}_roll28_minus_roll91"] = features[f"{prefix}_roll_mean_28"] - features[f"{prefix}_roll_mean_91"]

    margin = frame["Revenue"] / frame["COGS"].replace(0, np.nan)
    shifted_margin = margin.shift(1)
    for lag in [1, 7, 28, 365]:
        features[f"margin_lag_{lag}"] = margin.shift(lag)
    for window in [7, 28, 91, 365]:
        min_periods = max(3, window // 3)
        margin_roll = shifted_margin.rolling(window, min_periods=min_periods)
        features[f"margin_roll_mean_{window}"] = margin_roll.mean()
        features[f"margin_roll_std_{window}"] = margin_roll.std()

    revenue_shift = frame["Revenue"].astype(float).shift(1)
    cogs_shift = frame["COGS"].astype(float).shift(1)
    for window in [7, 28, 91, 365]:
        min_periods = max(3, window // 3)
        rev_roll = revenue_shift.rolling(window, min_periods=min_periods).mean()
        cogs_roll = cogs_shift.rolling(window, min_periods=min_periods).mean()
        features[f"rev_cogs_roll_ratio_{window}"] = rev_roll / cogs_roll.replace(0, np.nan)

    if include_current_cogs:
        current_cogs = frame["COGS"].astype(float)
        features["cogs_current"] = current_cogs
        features["cogs_current_log"] = np.log1p(current_cogs)
        for lag in [1, 7, 28, 365]:
            lag_col = f"cogs_lag_{lag}"
            features[f"cogs_current_div_{lag_col}"] = current_cogs / features[lag_col].replace(0, np.nan)
            features[f"cogs_current_minus_{lag_col}"] = current_cogs - features[lag_col]
        for window in [28, 91, 365]:
            roll_col = f"cogs_roll_mean_{window}"
            margin_col = f"margin_roll_mean_{window}"
            features[f"cogs_current_div_{roll_col}"] = current_cogs / features[roll_col].replace(0, np.nan)
            features[f"cogs_current_x_{margin_col}"] = current_cogs * features[margin_col]

    return features.replace([np.inf, -np.inf], np.nan)


def make_forecast_row(
    history: pd.DataFrame,
    forecast_date: pd.Timestamp,
    *,
    include_current_cogs: bool,
    current_cogs: float | None,
    origin: pd.Timestamp,
    feature_cols: list[str],
) -> pd.DataFrame:
    history = history.sort_values("date").set_index("date")
    row = calendar_features([forecast_date], origin).iloc[0].to_dict()

    for col in ["Revenue", "COGS"]:
        series = history[col].astype(float)
        prefix = col.lower()
        past = series[series.index < forecast_date]

        for lag in LAGS:
            row[f"{prefix}_lag_{lag}"] = series.get(
                forecast_date - pd.Timedelta(days=lag), np.nan
            )

        for window in ROLL_WINDOWS:
            min_periods = max(3, window // 3)
            values = past.tail(window)
            ok = len(values) >= min_periods
            row[f"{prefix}_roll_mean_{window}"] = values.mean() if ok else np.nan
            row[f"{prefix}_roll_median_{window}"] = values.median() if ok else np.nan
            row[f"{prefix}_roll_std_{window}"] = values.std() if ok else np.nan
            row[f"{prefix}_roll_min_{window}"] = values.min() if ok else np.nan
            row[f"{prefix}_roll_max_{window}"] = values.max() if ok else np.nan

        for span in EWM_SPANS:
            values = past.tail(max(3, span * 8))
            row[f"{prefix}_ewm_{span}"] = (
                values.ewm(span=span, adjust=False, min_periods=3).mean().iloc[-1]
                if len(values) >= 3
                else np.nan
            )

        denominator = row.get(f"{prefix}_lag_7", 0)
        row[f"{prefix}_lag1_div_lag7"] = (
            row.get(f"{prefix}_lag_1", np.nan) / denominator if denominator else np.nan
        )
        denominator = row.get(f"{prefix}_lag_28", 0)
        row[f"{prefix}_lag7_div_lag28"] = (
            row.get(f"{prefix}_lag_7", np.nan) / denominator if denominator else np.nan
        )
        denominator = row.get(f"{prefix}_lag_728", 0)
        row[f"{prefix}_lag364_div_lag728"] = (
            row.get(f"{prefix}_lag_364", np.nan) / denominator if denominator else np.nan
        )
        denominator = row.get(f"{prefix}_roll_mean_28", 0)
        row[f"{prefix}_roll7_div_roll28"] = (
            row.get(f"{prefix}_roll_mean_7", np.nan) / denominator if denominator else np.nan
        )
        denominator = row.get(f"{prefix}_roll_mean_91", 0)
        row[f"{prefix}_roll28_div_roll91"] = (
            row.get(f"{prefix}_roll_mean_28", np.nan) / denominator if denominator else np.nan
        )
        denominator = row.get(f"{prefix}_roll_mean_365", 0)
        row[f"{prefix}_roll91_div_roll365"] = (
            row.get(f"{prefix}_roll_mean_91", np.nan) / denominator if denominator else np.nan
        )
        denominator = row.get(f"{prefix}_roll_mean_365", 0)
        row[f"{prefix}_lag365_div_roll365"] = (
            row.get(f"{prefix}_lag_365", np.nan) / denominator if denominator else np.nan
        )
        row[f"{prefix}_lag1_minus_lag7"] = row.get(f"{prefix}_lag_1", np.nan) - row.get(f"{prefix}_lag_7", np.nan)
        row[f"{prefix}_roll7_minus_roll28"] = row.get(f"{prefix}_roll_mean_7", np.nan) - row.get(f"{prefix}_roll_mean_28", np.nan)
        row[f"{prefix}_roll28_minus_roll91"] = row.get(f"{prefix}_roll_mean_28", np.nan) - row.get(f"{prefix}_roll_mean_91", np.nan)

    margin = history["Revenue"] / history["COGS"].replace(0, np.nan)
    past_margin = margin[margin.index < forecast_date]
    for lag in [1, 7, 28, 365]:
        row[f"margin_lag_{lag}"] = margin.get(forecast_date - pd.Timedelta(days=lag), np.nan)
    for window in [7, 28, 91, 365]:
        min_periods = max(3, window // 3)
        values = past_margin.tail(window)
        ok = len(values) >= min_periods
        row[f"margin_roll_mean_{window}"] = values.mean() if ok else np.nan
        row[f"margin_roll_std_{window}"] = values.std() if ok else np.nan

    for window in [7, 28, 91, 365]:
        min_periods = max(3, window // 3)
        rev_values = history["Revenue"].astype(float)[history.index < forecast_date].tail(window)
        cogs_values = history["COGS"].astype(float)[history.index < forecast_date].tail(window)
        row[f"rev_cogs_roll_ratio_{window}"] = (
            rev_values.mean() / cogs_values.mean() if len(rev_values) >= min_periods and cogs_values.mean() else np.nan
        )

    if include_current_cogs:
        cogs_value = float(current_cogs or 0.0)
        row["cogs_current"] = cogs_value
        row["cogs_current_log"] = math.log1p(max(cogs_value, 0.0))
        for lag in [1, 7, 28, 365]:
            lag_col = f"cogs_lag_{lag}"
            denominator = row.get(lag_col, 0)
            row[f"cogs_current_div_{lag_col}"] = cogs_value / denominator if denominator else np.nan
            row[f"cogs_current_minus_{lag_col}"] = cogs_value - row.get(lag_col, np.nan)
        for window in [28, 91, 365]:
            roll_col = f"cogs_roll_mean_{window}"
            margin_col = f"margin_roll_mean_{window}"
            denominator = row.get(roll_col, 0)
            row[f"cogs_current_div_{roll_col}"] = cogs_value / denominator if denominator else np.nan
            row[f"cogs_current_x_{margin_col}"] = cogs_value * row.get(margin_col, np.nan)

    features = pd.DataFrame([row]).replace([np.inf, -np.inf], np.nan)
    for col in feature_cols:
        if col not in features.columns:
            features[col] = np.nan
    return features[feature_cols]


def seasonal_prediction(history: pd.DataFrame, forecast_date: pd.Timestamp, target: str, lag: int) -> float:
    series = history.set_index("date")[target].astype(float)
    base = series.get(forecast_date - pd.Timedelta(days=lag), np.nan)

    recent = history.tail(365)[target].sum()
    previous = history.iloc[-730:-365][target].sum() if len(history) >= 730 else recent
    growth = float(np.clip(recent / max(previous, 1.0), 0.82, 1.18))

    if pd.isna(base):
        base = series.get(forecast_date - pd.Timedelta(days=730), np.nan)
        power = 2
    else:
        power = 1
    if pd.isna(base):
        base = history.tail(365)[target].median()

    return max(1.0, float(base * (growth**power)))


def seasonal_blend_prediction(history: pd.DataFrame, forecast_date: pd.Timestamp, target: str) -> float:
    """Yearly seasonal blend using only train/recursive history.

    365-day lag preserves calendar date; 364/371-day lags preserve nearby
    weekday structure; 730-day lag anchors the level two years back. A clipped
    recent-over-previous annual growth factor lets the forecast adapt without
    exploding on one noisy year.
    """
    values = {
        364: seasonal_prediction(history, forecast_date, target, 364),
        365: seasonal_prediction(history, forecast_date, target, 365),
        371: seasonal_prediction(history, forecast_date, target, 371),
        730: seasonal_prediction(history, forecast_date, target, 730),
    }
    weights = {364: 0.20, 365: 0.40, 371: 0.10, 730: 0.30}
    return max(1.0, float(sum(values[lag] * weights[lag] for lag in weights)))


def doy_window_prediction(
    history: pd.DataFrame,
    forecast_date: pd.Timestamp,
    target: str,
    *,
    window: int = 10,
) -> float:
    """Median around the same day-of-year in recent years.

    This catches seasonal peaks that move by a few days. It uses only dates
    already present in history, which is train data plus recursive predictions
    produced earlier in the same forecast horizon.
    """
    series = history.sort_values("date").set_index("date")[target].astype(float)
    values = []
    weights = []
    for years_back, weight in [(1, 0.55), (2, 0.30), (3, 0.15)]:
        center = forecast_date - pd.DateOffset(years=years_back)
        local = [
            series.get(center + pd.Timedelta(days=offset), np.nan)
            for offset in range(-window, window + 1)
        ]
        local = [value for value in local if np.isfinite(value)]
        if local:
            values.append(float(np.median(local)))
            weights.append(weight)

    if not values:
        return seasonal_prediction(history, forecast_date, target, 365)

    pred = float(np.average(values, weights=weights))
    if len(history) >= 730:
        recent = history.tail(365)[target].median()
        previous = history.iloc[-730:-365][target].median()
        growth = float(np.clip(recent / max(previous, 1.0), 0.90, 1.12))
        pred *= growth
    return max(1.0, pred)


def local_trend_prediction(
    history: pd.DataFrame,
    forecast_date: pd.Timestamp,
    target: str,
    *,
    window: int = 12,
) -> float:
    """Recent-year local trend around the same calendar day.

    Instead of relying on one exact lag, this fits a tiny weighted linear trend
    on yearly local medians. It is conservative-clipped against recent local
    history so RMSE does not blow up on extrapolation.
    """
    hist = history.sort_values("date")
    yearly = []
    for year in sorted(hist["date"].dt.year.unique()):
        last_day = pd.Timestamp(year=year, month=forecast_date.month, day=1).days_in_month
        center = pd.Timestamp(year=year, month=forecast_date.month, day=min(forecast_date.day, last_day))
        mask = (hist["date"] >= center - pd.Timedelta(days=window)) & (hist["date"] <= center + pd.Timedelta(days=window))
        vals = hist.loc[mask, target].astype(float).dropna()
        if len(vals) >= 5:
            yearly.append((float(year), float(vals.median())))

    if len(yearly) < 3:
        return doy_window_prediction(history, forecast_date, target, window=window)

    years = np.asarray([row[0] for row in yearly], dtype=float)
    medians = np.asarray([row[1] for row in yearly], dtype=float)
    recent_weight = np.linspace(0.35, 1.0, len(yearly))
    slope, intercept = np.polyfit(years, medians, deg=1, w=recent_weight)
    pred = float(slope * forecast_date.year + intercept)
    recent_values = medians[-min(4, len(medians)) :]
    low, high = np.quantile(recent_values, [0.10, 0.90])
    pred = float(np.clip(pred, low * 0.85, high * 1.20))
    return max(1.0, pred)

def margin_based_prediction(
    history: pd.DataFrame,
    forecast_date: pd.Timestamp,
    cogs_pred: float,
    *,
    window: int | None = None,
    month_specific: bool = False,
) -> float:
    past = history[history["date"] < forecast_date].copy()
    margin = past["Revenue"].astype(float) / past["COGS"].replace(0, np.nan).astype(float)
    if month_specific:
        values = margin[past["date"].dt.month == forecast_date.month].dropna()
    elif window is not None:
        values = margin.tail(window).dropna()
    else:
        values = margin.dropna()

    if values.empty:
        values = margin.tail(365).dropna()
    if values.empty:
        margin_value = 1.2
    else:
        margin_value = float(np.clip(values.median(), 1.02, 1.60))
    return max(1.0, float(cogs_pred * margin_value))


def _clip_forecast_values(values: np.ndarray, history: pd.DataFrame, target: str) -> np.ndarray:
    recent = history.tail(1095)[target].astype(float)
    low = max(1.0, float(recent.quantile(0.001) * 0.45))
    high = float(recent.quantile(0.999) * 1.80)
    return np.clip(np.asarray(values, dtype=float), low, high)


def _fallback_ts_values(history: pd.DataFrame, future_dates: pd.Series, target: str) -> np.ndarray:
    return np.asarray([seasonal_prediction(history, date, target, 365) for date in pd.to_datetime(future_dates)], dtype=float)


def statsmodels_time_series_forecasts(
    history: pd.DataFrame,
    future_dates: pd.Series,
    target: str,
) -> pd.DataFrame:
    """Specialized time-series candidates using train history only.

    These models are fit once per fold on the available training segment. They
    never see test Revenue/COGS. They provide classical time-series views that
    complement tree/linear supervised lag models.
    """
    future_dates = pd.to_datetime(pd.Series(future_dates)).reset_index(drop=True)
    prefix = "rev" if target == "Revenue" else "cogs"
    out = pd.DataFrame({"date": future_dates})
    fallback = _fallback_ts_values(history, future_dates, target)

    if not USE_STATSMODELS_CANDIDATES or not STATSMODELS_AVAILABLE or len(history) < 730:
        out[f"{prefix}_ts_ets"] = fallback
        out[f"{prefix}_ts_theta"] = fallback
        out[f"{prefix}_ts_sarimax_weekly"] = fallback
        return out

    series = (
        history.sort_values("date")
        .set_index("date")[target]
        .astype(float)
        .asfreq("D")
        .interpolate(limit_direction="both")
    )
    y = series.tail(min(STATSMODELS_TAIL_DAYS, len(series)))
    steps = len(future_dates)

    def safe_predict(name: str, fit_predict) -> np.ndarray:
        try:
            pred = np.asarray(fit_predict(), dtype=float)
            if len(pred) != steps or not np.isfinite(pred).all():
                raise ValueError(f"bad forecast from {name}")
            return _clip_forecast_values(pred, history, target)
        except Exception:
            return fallback

    ets_pred = safe_predict(
        "ETS",
        lambda: ExponentialSmoothing(
            y,
            trend="add",
            damped_trend=True,
            seasonal=None,
            initialization_method="estimated",
        ).fit(optimized=True).forecast(steps),
    )
    theta_pred = safe_predict(
        "Theta",
        lambda: ThetaModel(y, period=365).fit().forecast(steps),
    )

    y_sarimax = y.tail(min(SARIMAX_TAIL_DAYS, len(y)))
    sarimax_pred = safe_predict(
        "SARIMAX_weekly",
        lambda: SARIMAX(
            y_sarimax,
            order=(1, 1, 1),
            seasonal_order=(1, 0, 1, 7),
            enforce_stationarity=False,
            enforce_invertibility=False,
        ).fit(disp=False, maxiter=50).forecast(steps),
    )

    out[f"{prefix}_ts_ets"] = ets_pred
    out[f"{prefix}_ts_theta"] = theta_pred
    out[f"{prefix}_ts_sarimax_weekly"] = sarimax_pred
    return out


## 1.4. Định nghĩa mô hình và huấn luyện

Pipeline dùng ensemble nhiều họ mô hình để cân bằng bias/variance: LightGBM, XGBoost, HistGradientBoosting và Ridge. Có hai nhóm objective song song: MAE/absolute-error để kéo MAE xuống, và L2/squared-error để giữ RMSE thấp và R2 cao. Có hai thang target song song: `log1p` giúp ổn định outlier, còn raw-target giữ khả năng tối ưu trực tiếp trên đơn vị doanh thu.

Revenue model có hai nhánh: nhánh dùng `COGS` cùng ngày của train, và nhánh không dùng `COGS` cùng ngày. Khi dự báo test, `COGS` cùng ngày là giá trị do model COGS dự báo nội sinh, không phải COGS trong template/test. Thiết kế này tận dụng quan hệ Revenue-COGS ở train nhưng vẫn kiểm soát leakage ở test.


In [ ]:
def _lgb_params() -> dict:
    return {
        "objective": "mae",
        "n_estimators": 500,
        "learning_rate": 0.035,
        "num_leaves": 31,
        "max_depth": 6,
        "min_child_samples": 18,
        "subsample": 0.90,
        "colsample_bytree": 0.90,
        "reg_alpha": 0.05,
        "reg_lambda": 1.50,
        "random_state": SEED,
        "verbose": -1,
        "n_jobs": 2,
    }


def _lgb_l2_params() -> dict:
    params = _lgb_params()
    params.update({
        "objective": "regression",
        "n_estimators": 650,
        "learning_rate": 0.030,
        "num_leaves": 39,
        "max_depth": 7,
        "reg_lambda": 2.0,
    })
    return params


def _xgb_model() -> XGBRegressor:
    return XGBRegressor(
        n_estimators=450,
        learning_rate=0.035,
        max_depth=4,
        min_child_weight=3,
        subsample=0.90,
        colsample_bytree=0.90,
        reg_lambda=2.0,
        reg_alpha=0.05,
        random_state=SEED,
        objective="reg:absoluteerror",
        eval_metric="mae",
        tree_method="hist",
        n_jobs=2,
    )


def _xgb_l2_model() -> XGBRegressor:
    return XGBRegressor(
        n_estimators=650,
        learning_rate=0.030,
        max_depth=5,
        min_child_weight=3,
        subsample=0.90,
        colsample_bytree=0.90,
        reg_lambda=2.0,
        reg_alpha=0.02,
        random_state=SEED,
        objective="reg:squarederror",
        eval_metric="rmse",
        tree_method="hist",
        n_jobs=2,
    )


def _hgb_model() -> HistGradientBoostingRegressor:
    return HistGradientBoostingRegressor(
        loss="absolute_error",
        max_iter=420,
        learning_rate=0.035,
        max_leaf_nodes=31,
        max_depth=7,
        min_samples_leaf=18,
        l2_regularization=0.01,
        random_state=SEED,
    )


def _hgb_l2_model() -> HistGradientBoostingRegressor:
    return HistGradientBoostingRegressor(
        loss="squared_error",
        max_iter=520,
        learning_rate=0.035,
        max_leaf_nodes=63,
        max_depth=8,
        min_samples_leaf=18,
        l2_regularization=0.10,
        random_state=SEED,
    )


def fit_models(train: pd.DataFrame) -> FittedBundle:
    train = train.sort_values("date").reset_index(drop=True)
    origin = train["date"].min()

    cogs_features = make_training_features(train, include_current_cogs=False, origin=origin)
    revenue_features = make_training_features(train, include_current_cogs=True, origin=origin)
    valid_mask = cogs_features["revenue_lag_730"].notna() & cogs_features["cogs_lag_730"].notna()

    x_cogs = cogs_features.loc[valid_mask].copy()
    x_revenue = revenue_features.loc[valid_mask].copy()
    y_cogs = np.log1p(train.loc[valid_mask, "COGS"].values)
    y_revenue = np.log1p(train.loc[valid_mask, "Revenue"].values)
    y_cogs_raw = train.loc[valid_mask, "COGS"].values.astype(float)
    y_revenue_raw = train.loc[valid_mask, "Revenue"].values.astype(float)
    y_margin = np.log(
        (train.loc[valid_mask, "Revenue"].values + 1.0)
        / (train.loc[valid_mask, "COGS"].values + 1.0)
    )

    cogs_medians = x_cogs.median(numeric_only=True).fillna(0.0)
    revenue_medians = x_revenue.median(numeric_only=True).fillna(0.0)
    x_cogs = x_cogs.fillna(cogs_medians).fillna(0.0)
    x_revenue = x_revenue.fillna(revenue_medians).fillna(0.0)

    models = {}
    for prefix, x_matrix, y_values in [
        ("cogs", x_cogs, y_cogs),
        ("rev", x_revenue, y_revenue),
    ]:
        models[f"{prefix}_lgb"] = LGBMRegressor(**_lgb_params()).fit(x_matrix, y_values)
        models[f"{prefix}_xgb"] = _xgb_model().fit(x_matrix, y_values)
        models[f"{prefix}_hgb"] = _hgb_model().fit(x_matrix, y_values)
        models[f"{prefix}_ridge"] = make_pipeline(
            StandardScaler(), Ridge(alpha=50.0)
        ).fit(x_matrix, y_values)
        # L2/log-target variants improve RMSE and R2 candidates for the final multi-metric ensemble.
        models[f"{prefix}_lgb_l2"] = LGBMRegressor(**_lgb_l2_params()).fit(x_matrix, y_values)
        models[f"{prefix}_xgb_l2"] = _xgb_l2_model().fit(x_matrix, y_values)
        models[f"{prefix}_hgb_l2"] = _hgb_l2_model().fit(x_matrix, y_values)

    raw_params = _lgb_params()
    raw_params.update({"n_estimators": 700, "learning_rate": 0.025, "num_leaves": 31})
    models["cogs_lgb_raw"] = LGBMRegressor(**raw_params).fit(x_cogs, y_cogs_raw)
    models["rev_lgb_raw"] = LGBMRegressor(**raw_params).fit(x_revenue, y_revenue_raw)
    models["cogs_xgb_raw"] = _xgb_model().fit(x_cogs, y_cogs_raw)
    models["rev_xgb_raw"] = _xgb_model().fit(x_revenue, y_revenue_raw)
    models["cogs_hgb_raw"] = _hgb_model().fit(x_cogs, y_cogs_raw)
    models["rev_hgb_raw"] = _hgb_model().fit(x_revenue, y_revenue_raw)

    # Revenue models without current COGS reduce mismatch between train-time true COGS
    # and test-time internally forecasted COGS.
    models["rev_lgb_nocogs"] = LGBMRegressor(**_lgb_params()).fit(x_cogs, y_revenue)
    models["rev_xgb_nocogs"] = _xgb_model().fit(x_cogs, y_revenue)
    models["rev_hgb_nocogs"] = _hgb_model().fit(x_cogs, y_revenue)
    models["rev_ridge_nocogs"] = make_pipeline(StandardScaler(), Ridge(alpha=50.0)).fit(x_cogs, y_revenue)
    models["rev_lgb_raw_nocogs"] = LGBMRegressor(**raw_params).fit(x_cogs, y_revenue_raw)
    models["rev_xgb_raw_nocogs"] = _xgb_model().fit(x_cogs, y_revenue_raw)
    models["rev_hgb_raw_nocogs"] = _hgb_model().fit(x_cogs, y_revenue_raw)

    ratio_params = _lgb_params()
    ratio_params.update({"n_estimators": 400, "num_leaves": 23})
    models["ratio_lgb"] = LGBMRegressor(**ratio_params).fit(x_revenue, y_margin)

    return FittedBundle(
        models=models,
        cogs_cols=x_cogs.columns.tolist(),
        revenue_cols=x_revenue.columns.tolist(),
        cogs_medians=cogs_medians,
        revenue_medians=revenue_medians,
        origin=origin,
    )


def forecast(
    train: pd.DataFrame,
    future_dates: pd.Series,
    *,
    cogs_weights: np.ndarray | None = None,
    revenue_weights: np.ndarray | None = None,
) -> tuple[pd.DataFrame, FittedBundle]:
    if cogs_weights is None:
        cogs_weights = DEFAULT_COGS_WEIGHTS
    if revenue_weights is None:
        revenue_weights = DEFAULT_REVENUE_WEIGHTS

    bundle = fit_models(train)
    cogs_ts_candidates = statsmodels_time_series_forecasts(train, future_dates, "COGS").set_index("date")
    revenue_ts_candidates = statsmodels_time_series_forecasts(train, future_dates, "Revenue").set_index("date")
    history = train[["date", "Revenue", "COGS"]].copy()
    rows = []

    for forecast_date in pd.to_datetime(future_dates):
        x_cogs = make_forecast_row(
            history,
            forecast_date,
            include_current_cogs=False,
            current_cogs=None,
            origin=bundle.origin,
            feature_cols=bundle.cogs_cols,
        ).fillna(bundle.cogs_medians).fillna(0.0)

        cogs_candidates = {
            "cogs_lgb": float(np.expm1(bundle.models["cogs_lgb"].predict(x_cogs)[0])),
            "cogs_xgb": float(np.expm1(bundle.models["cogs_xgb"].predict(x_cogs)[0])),
            "cogs_hgb": float(np.expm1(bundle.models["cogs_hgb"].predict(x_cogs)[0])),
            "cogs_ridge": float(np.expm1(bundle.models["cogs_ridge"].predict(x_cogs)[0])),
            "cogs_lgb_l2": float(np.expm1(bundle.models["cogs_lgb_l2"].predict(x_cogs)[0])),
            "cogs_xgb_l2": float(np.expm1(bundle.models["cogs_xgb_l2"].predict(x_cogs)[0])),
            "cogs_hgb_l2": float(np.expm1(bundle.models["cogs_hgb_l2"].predict(x_cogs)[0])),
            "cogs_lgb_raw": float(bundle.models["cogs_lgb_raw"].predict(x_cogs)[0]),
            "cogs_xgb_raw": float(bundle.models["cogs_xgb_raw"].predict(x_cogs)[0]),
            "cogs_hgb_raw": float(bundle.models["cogs_hgb_raw"].predict(x_cogs)[0]),
            "cogs_season364": seasonal_prediction(history, forecast_date, "COGS", 364),
            "cogs_season365": seasonal_prediction(history, forecast_date, "COGS", 365),
            "cogs_season371": seasonal_prediction(history, forecast_date, "COGS", 371),
            "cogs_season730": seasonal_prediction(history, forecast_date, "COGS", 730),
            "cogs_season_blend": seasonal_blend_prediction(history, forecast_date, "COGS"),
            "cogs_doy_window": doy_window_prediction(history, forecast_date, "COGS"),
            "cogs_local_trend": local_trend_prediction(history, forecast_date, "COGS"),
            "cogs_ts_ets": float(cogs_ts_candidates.loc[forecast_date, "cogs_ts_ets"]),
            "cogs_ts_theta": float(cogs_ts_candidates.loc[forecast_date, "cogs_ts_theta"]),
            "cogs_ts_sarimax_weekly": float(cogs_ts_candidates.loc[forecast_date, "cogs_ts_sarimax_weekly"]),
        }
        cogs_pred = float(np.dot(cogs_weights, [cogs_candidates[col] for col in COGS_MODEL_COLUMNS]))
        cogs_pred = max(1.0, cogs_pred)

        x_revenue = make_forecast_row(
            history,
            forecast_date,
            include_current_cogs=True,
            current_cogs=cogs_pred,
            origin=bundle.origin,
            feature_cols=bundle.revenue_cols,
        ).fillna(bundle.revenue_medians).fillna(0.0)

        revenue_candidates = {
            "rev_lgb": float(np.expm1(bundle.models["rev_lgb"].predict(x_revenue)[0])),
            "rev_xgb": float(np.expm1(bundle.models["rev_xgb"].predict(x_revenue)[0])),
            "rev_hgb": float(np.expm1(bundle.models["rev_hgb"].predict(x_revenue)[0])),
            "rev_ridge": float(np.expm1(bundle.models["rev_ridge"].predict(x_revenue)[0])),
            "rev_lgb_l2": float(np.expm1(bundle.models["rev_lgb_l2"].predict(x_revenue)[0])),
            "rev_xgb_l2": float(np.expm1(bundle.models["rev_xgb_l2"].predict(x_revenue)[0])),
            "rev_hgb_l2": float(np.expm1(bundle.models["rev_hgb_l2"].predict(x_revenue)[0])),
            "rev_ratio_lgb": cogs_pred
            * float(np.exp(bundle.models["ratio_lgb"].predict(x_revenue)[0])),
            "rev_lgb_raw": float(bundle.models["rev_lgb_raw"].predict(x_revenue)[0]),
            "rev_xgb_raw": float(bundle.models["rev_xgb_raw"].predict(x_revenue)[0]),
            "rev_hgb_raw": float(bundle.models["rev_hgb_raw"].predict(x_revenue)[0]),
            "rev_lgb_nocogs": float(np.expm1(bundle.models["rev_lgb_nocogs"].predict(x_cogs)[0])),
            "rev_xgb_nocogs": float(np.expm1(bundle.models["rev_xgb_nocogs"].predict(x_cogs)[0])),
            "rev_hgb_nocogs": float(np.expm1(bundle.models["rev_hgb_nocogs"].predict(x_cogs)[0])),
            "rev_ridge_nocogs": float(np.expm1(bundle.models["rev_ridge_nocogs"].predict(x_cogs)[0])),
            "rev_lgb_raw_nocogs": float(bundle.models["rev_lgb_raw_nocogs"].predict(x_cogs)[0]),
            "rev_xgb_raw_nocogs": float(bundle.models["rev_xgb_raw_nocogs"].predict(x_cogs)[0]),
            "rev_hgb_raw_nocogs": float(bundle.models["rev_hgb_raw_nocogs"].predict(x_cogs)[0]),
            "rev_margin_roll_28": margin_based_prediction(history, forecast_date, cogs_pred, window=28),
            "rev_margin_roll_91": margin_based_prediction(history, forecast_date, cogs_pred, window=91),
            "rev_margin_roll_365": margin_based_prediction(history, forecast_date, cogs_pred, window=365),
            "rev_margin_month": margin_based_prediction(history, forecast_date, cogs_pred, month_specific=True),
            "rev_season364": seasonal_prediction(history, forecast_date, "Revenue", 364),
            "rev_season365": seasonal_prediction(history, forecast_date, "Revenue", 365),
            "rev_season371": seasonal_prediction(history, forecast_date, "Revenue", 371),
            "rev_season730": seasonal_prediction(history, forecast_date, "Revenue", 730),
            "rev_season_blend": seasonal_blend_prediction(history, forecast_date, "Revenue"),
            "rev_doy_window": doy_window_prediction(history, forecast_date, "Revenue"),
            "rev_local_trend": local_trend_prediction(history, forecast_date, "Revenue"),
            "rev_ts_ets": float(revenue_ts_candidates.loc[forecast_date, "rev_ts_ets"]),
            "rev_ts_theta": float(revenue_ts_candidates.loc[forecast_date, "rev_ts_theta"]),
            "rev_ts_sarimax_weekly": float(revenue_ts_candidates.loc[forecast_date, "rev_ts_sarimax_weekly"]),
        }
        revenue_pred = float(
            np.dot(revenue_weights, [revenue_candidates[col] for col in REVENUE_MODEL_COLUMNS])
        )

        recent_revenue = history.tail(1095)["Revenue"]
        low_clip = max(0.0, float(recent_revenue.quantile(0.001) * 0.55))
        high_clip = float(recent_revenue.quantile(0.999) * 1.60)
        revenue_pred = float(np.clip(revenue_pred, low_clip, high_clip))

        row = {
            "date": forecast_date,
            "cogs_pred": cogs_pred,
            "revenue_pred": revenue_pred,
            **cogs_candidates,
            **revenue_candidates,
        }
        rows.append(row)
        history = pd.concat(
            [
                history,
                pd.DataFrame(
                    [{"date": forecast_date, "Revenue": revenue_pred, "COGS": cogs_pred}]
                ),
            ],
            ignore_index=True,
        )

    return pd.DataFrame(rows), bundle


## 1.5. Backtest, tuning đa metric, report và xuất submission

Cross-validation được chia theo chiều thời gian, trong đó các fold gần 2022 được ưu tiên vì test bắt đầu ngay sau 2022-12-31. Notebook lưu OOF prediction của từng candidate, sau đó tìm trọng số ensemble bằng quy trình MAE-first: trước hết giữ các nghiệm có MAE gần tốt nhất, rồi chọn nghiệm có objective tổng hợp MAE/RMSE/R2 tốt nhất.

Sau blend, correction theo tháng được học trên fold gần nhất với shrink và clip để sửa bias mùa vụ còn sót lại. Đây không phải scale hack public score; correction được tính hoàn toàn từ train backtest và bị giới hạn biên độ.


In [ ]:
OOF_CANDIDATE_PREDICTIONS = pd.DataFrame()
MAE_TUNED_WEIGHT_TABLE = pd.DataFrame()


def _apply_weight_caps(weights: np.ndarray, candidate_cols: list[str]) -> np.ndarray:
    caps = np.asarray([MODEL_WEIGHT_CAPS.get(col, DEFAULT_SINGLE_MODEL_CAP) for col in candidate_cols], dtype=float)
    if caps.sum() < 1.0:
        caps = caps / caps.sum()

    raw = np.maximum(np.asarray(weights, dtype=float), 0.0)
    if raw.sum() <= 0 or not np.isfinite(raw).all():
        raw = np.ones(len(candidate_cols), dtype=float)

    result = np.zeros(len(candidate_cols), dtype=float)
    active = np.ones(len(candidate_cols), dtype=bool)
    remaining = 1.0

    for _ in range(len(candidate_cols) + 1):
        active_idx = np.where(active)[0]
        if len(active_idx) == 0 or remaining <= 1e-12:
            break

        active_raw = raw[active_idx].copy()
        if active_raw.sum() <= 0 or not np.isfinite(active_raw).all():
            active_raw = np.ones(len(active_idx), dtype=float)

        proposed = active_raw / active_raw.sum() * remaining
        over = proposed > caps[active_idx] + 1e-12
        if not over.any():
            result[active_idx] = proposed
            remaining = 0.0
            break

        capped_idx = active_idx[over]
        result[capped_idx] = caps[capped_idx]
        remaining = max(0.0, 1.0 - result.sum())
        active[capped_idx] = False
        raw[capped_idx] = 0.0

    if result.sum() <= 0 or not np.isfinite(result).all():
        result = np.minimum(np.ones(len(candidate_cols), dtype=float) / len(candidate_cols), caps)
    result = result / result.sum()
    return result


def mae_weight_search(
    oof: pd.DataFrame,
    candidate_cols: list[str],
    target_col: str,
    *,
    n_random: int = 20000,
    seed: int = SEED,
    fold_weights: dict[str, float] | None = None,
) -> tuple[np.ndarray, pd.DataFrame]:
    """Find convex ensemble weights with a MAE-first multi-metric objective.

    Kaggle currently displays MAE, so the first filter keeps only solutions
    whose weighted fold MAE is within ENSEMBLE_MAE_TOLERANCE of the best MAE.
    Among those near-best-MAE candidates, the chosen weights minimize a
    normalized objective that also rewards lower RMSE and higher R2.
    """
    rng = np.random.default_rng(seed)
    y = oof[target_col].to_numpy(dtype=float)
    pred_matrix = oof[candidate_cols].to_numpy(dtype=float)
    folds = oof["fold"].to_numpy()
    unique_folds = np.unique(folds)
    if fold_weights is None:
        fold_weights = {fold: 1.0 for fold in unique_folds}
    active_folds = [fold for fold in unique_folds if fold_weights.get(fold, 0.0) > 0]
    total_fold_weight = sum(float(fold_weights.get(fold, 0.0)) for fold in active_folds)

    y_scale = float(np.mean(np.abs(y - np.median(y))))
    y_scale = max(y_scale, 1.0)

    def score(weights: np.ndarray) -> dict[str, float]:
        pred = pred_matrix @ weights
        fold_metrics = []
        for fold in active_folds:
            mask = folds == fold
            yt = y[mask]
            yp = pred[mask]
            fold_metrics.append(
                (
                    float(fold_weights.get(fold, 0.0)),
                    float(mean_absolute_error(yt, yp)),
                    float(np.sqrt(mean_squared_error(yt, yp))),
                    float(r2_score(yt, yp)),
                )
            )

        weighted_mae = sum(w * mae for w, mae, _, _ in fold_metrics) / max(total_fold_weight, 1e-12)
        weighted_rmse = sum(w * rmse for w, _, rmse, _ in fold_metrics) / max(total_fold_weight, 1e-12)
        weighted_r2 = sum(w * r2 for w, _, _, r2 in fold_metrics) / max(total_fold_weight, 1e-12)
        overall_mae = float(mean_absolute_error(y, pred))
        overall_rmse = float(np.sqrt(mean_squared_error(y, pred)))
        overall_r2 = float(r2_score(y, pred))
        objective = (
            ENSEMBLE_OBJECTIVE_WEIGHTS["MAE"] * (weighted_mae / y_scale)
            + ENSEMBLE_OBJECTIVE_WEIGHTS["RMSE"] * (weighted_rmse / y_scale)
            + ENSEMBLE_OBJECTIVE_WEIGHTS["R2"] * (1.0 - weighted_r2)
        )
        return {
            "objective": float(objective),
            "mean_fold_MAE": float(weighted_mae),
            "mean_fold_RMSE": float(weighted_rmse),
            "mean_fold_R2": float(weighted_r2),
            "MAE": overall_mae,
            "RMSE": overall_rmse,
            "R2": overall_r2,
        }

    candidates = []
    for i in range(len(candidate_cols)):
        w = np.zeros(len(candidate_cols))
        w[i] = 1.0
        candidates.append(_apply_weight_caps(w, candidate_cols))
    candidates.append(_apply_weight_caps(np.ones(len(candidate_cols)) / len(candidate_cols), candidate_cols))

    for alpha in [0.20, 0.35, 0.60, 1.00, 2.00, 4.00]:
        for _ in range(max(1, n_random // 6)):
            candidates.append(_apply_weight_caps(rng.dirichlet(np.ones(len(candidate_cols)) * alpha), candidate_cols))

    rows = []
    for weights in candidates:
        metric_values = score(weights)
        rows.append(
            {
                **metric_values,
                **{f"w_{col}": float(w) for col, w in zip(candidate_cols, weights)},
            }
        )

    detail = pd.DataFrame(rows)
    best_mae = float(detail["mean_fold_MAE"].min())
    eligible = detail[detail["mean_fold_MAE"] <= best_mae * (1.0 + ENSEMBLE_MAE_TOLERANCE)].copy()
    if eligible.empty:
        eligible = detail.copy()
    best_idx = eligible.sort_values(["objective", "mean_fold_MAE", "mean_fold_RMSE"], ascending=[True, True, True]).index[0]
    best_weights = detail.loc[best_idx, [f"w_{col}" for col in candidate_cols]].to_numpy(dtype=float)
    best_weights = best_weights / best_weights.sum()

    detail = detail.sort_values(["objective", "mean_fold_MAE", "mean_fold_RMSE"], ascending=[True, True, True]).reset_index(drop=True)
    return best_weights, detail

def run_backtests(sales: pd.DataFrame) -> pd.DataFrame:
    global DEFAULT_COGS_WEIGHTS, DEFAULT_REVENUE_WEIGHTS, MONTH_CORRECTION_FACTORS
    global OOF_CANDIDATE_PREDICTIONS, MAE_TUNED_WEIGHT_TABLE

    folds = [
        ("long_2020_2021H1", pd.Timestamp("2019-12-31"), 548),
        ("long_2021_2022H1", pd.Timestamp("2020-12-31"), 548),
        ("calendar_2022", pd.Timestamp("2021-12-31"), 365),
        ("recent_2021H2_2022", pd.Timestamp("2021-07-01"), 548),
        ("short_recent_2022H2", pd.Timestamp("2022-06-30"), 184),
    ]
    if FAST_MODE:
        folds = [
            ("calendar_2022", pd.Timestamp("2021-12-31"), 365),
            ("short_recent_2022H2", pd.Timestamp("2022-06-30"), 184),
        ]
    rows = []
    oof_rows = []

    for fold_name, train_end, horizon in folds:
        train = sales[sales["date"] <= train_end].copy()
        validation = sales[sales["date"] > train_end].head(horizon).copy()
        if len(validation) < horizon:
            continue

        predictions, _ = forecast(train, validation["date"])
        merged = predictions.copy()
        merged["fold"] = fold_name
        merged["actual_revenue"] = validation["Revenue"].to_numpy(dtype=float)
        merged["actual_cogs"] = validation["COGS"].to_numpy(dtype=float)
        oof_rows.append(merged)

        y_true = validation["Revenue"].values
        for col in ["revenue_pred", *REVENUE_MODEL_COLUMNS]:
            metric_values = _metric_dict(y_true, predictions[col].values)
            model_name = "default_blend" if col == "revenue_pred" else col
            rows.append(
                {
                    "fold": f"{fold_name}:{model_name}",
                    "train_end": train_end.date().isoformat(),
                    "val_start": validation["date"].min().date().isoformat(),
                    "val_end": validation["date"].max().date().isoformat(),
                    **metric_values,
                }
            )

    OOF_CANDIDATE_PREDICTIONS = pd.concat(oof_rows, ignore_index=True)
    OOF_CANDIDATE_PREDICTIONS.to_csv(REPORT_DIR / "oof_candidate_predictions.csv", index=False)

    tuned_cogs_weights, cogs_search = mae_weight_search(
        OOF_CANDIDATE_PREDICTIONS,
        COGS_MODEL_COLUMNS,
        "actual_cogs",
        n_random=8000 if FAST_MODE else 26000,
        seed=SEED + 17,
        fold_weights=FINAL_TUNING_FOLD_WEIGHTS,
    )
    tuned_revenue_weights, revenue_search = mae_weight_search(
        OOF_CANDIDATE_PREDICTIONS,
        REVENUE_MODEL_COLUMNS,
        "actual_revenue",
        n_random=14000 if FAST_MODE else 64000,
        seed=SEED + 29,
        fold_weights=FINAL_TUNING_FOLD_WEIGHTS,
    )

    DEFAULT_COGS_WEIGHTS = tuned_cogs_weights / tuned_cogs_weights.sum()
    DEFAULT_REVENUE_WEIGHTS = tuned_revenue_weights / tuned_revenue_weights.sum()

    cogs_search.head(200).to_csv(REPORT_DIR / "mae_weight_search_cogs_top200.csv", index=False)
    revenue_search.head(200).to_csv(REPORT_DIR / "mae_weight_search_revenue_top200.csv", index=False)

    MAE_TUNED_WEIGHT_TABLE = pd.DataFrame(
        [{"target": "COGS", "model": col, "weight": float(w)} for col, w in zip(COGS_MODEL_COLUMNS, DEFAULT_COGS_WEIGHTS)]
        + [{"target": "Revenue", "model": col, "weight": float(w)} for col, w in zip(REVENUE_MODEL_COLUMNS, DEFAULT_REVENUE_WEIGHTS)]
    )
    MAE_TUNED_WEIGHT_TABLE.to_csv(REPORT_DIR / "model_weights.csv", index=False)

    tuned_pred = OOF_CANDIDATE_PREDICTIONS[REVENUE_MODEL_COLUMNS].to_numpy(dtype=float) @ DEFAULT_REVENUE_WEIGHTS
    OOF_CANDIDATE_PREDICTIONS["mae_tuned_blend"] = tuned_pred

    recent_mask = OOF_CANDIDATE_PREDICTIONS["fold"].eq("recent_2021H2_2022")
    recent_oof = OOF_CANDIDATE_PREDICTIONS.loc[recent_mask].copy()
    recent_oof["month"] = pd.to_datetime(recent_oof["date"]).dt.month
    recent_oof["actual_over_pred"] = recent_oof["actual_revenue"] / recent_oof["mae_tuned_blend"].replace(0, np.nan)
    raw_month_factor = recent_oof.groupby("month")["actual_over_pred"].median()
    MONTH_CORRECTION_FACTORS = (
        1.0 + (raw_month_factor - 1.0) * MONTH_CORRECTION_SHRINK
    ).clip(MONTH_CORRECTION_CLIP[0], MONTH_CORRECTION_CLIP[1]).sort_index()
    MONTH_CORRECTION_FACTORS.rename("correction_factor").to_csv(REPORT_DIR / "month_correction_factors.csv")

    OOF_CANDIDATE_PREDICTIONS["month"] = pd.to_datetime(OOF_CANDIDATE_PREDICTIONS["date"]).dt.month
    OOF_CANDIDATE_PREDICTIONS["mae_tuned_month_corrected"] = (
        OOF_CANDIDATE_PREDICTIONS["mae_tuned_blend"]
        * OOF_CANDIDATE_PREDICTIONS["month"].map(MONTH_CORRECTION_FACTORS).fillna(1.0)
    )
    OOF_CANDIDATE_PREDICTIONS.to_csv(REPORT_DIR / "oof_candidate_predictions.csv", index=False)

    for fold_name, fold_data in OOF_CANDIDATE_PREDICTIONS.groupby("fold"):
        rows.append(
            {
                "fold": fold_name,
                "train_end": "MAE_tuned",
                "val_start": str(fold_data["date"].min().date()),
                "val_end": str(fold_data["date"].max().date()),
                **_metric_dict(fold_data["actual_revenue"].values, fold_data["mae_tuned_blend"].values),
            }
        )
        rows.append(
            {
                "fold": f"{fold_name}_month_corrected",
                "train_end": "MAE_tuned_month_corrected",
                "val_start": str(fold_data["date"].min().date()),
                "val_end": str(fold_data["date"].max().date()),
                **_metric_dict(fold_data["actual_revenue"].values, fold_data["mae_tuned_month_corrected"].values),
            }
        )

    return pd.DataFrame(rows)


def load_existing_mae_weights() -> None:
    global DEFAULT_COGS_WEIGHTS, DEFAULT_REVENUE_WEIGHTS, MAE_TUNED_WEIGHT_TABLE, MONTH_CORRECTION_FACTORS
    path = REPORT_DIR / "model_weights.csv"
    if not path.exists():
        return
    weights = pd.read_csv(path)
    if set(weights["target"]) >= {"COGS", "Revenue"}:
        cogs = weights[weights["target"] == "COGS"].set_index("model").reindex(COGS_MODEL_COLUMNS)["weight"]
        revenue = weights[weights["target"] == "Revenue"].set_index("model").reindex(REVENUE_MODEL_COLUMNS)["weight"]
        if cogs.notna().all() and revenue.notna().all():
            DEFAULT_COGS_WEIGHTS = cogs.to_numpy(dtype=float)
            DEFAULT_REVENUE_WEIGHTS = revenue.to_numpy(dtype=float)
            DEFAULT_COGS_WEIGHTS = DEFAULT_COGS_WEIGHTS / DEFAULT_COGS_WEIGHTS.sum()
            DEFAULT_REVENUE_WEIGHTS = DEFAULT_REVENUE_WEIGHTS / DEFAULT_REVENUE_WEIGHTS.sum()
            MAE_TUNED_WEIGHT_TABLE = weights.copy()
    correction_path = REPORT_DIR / "month_correction_factors.csv"
    if correction_path.exists():
        correction = pd.read_csv(correction_path, index_col=0).iloc[:, 0]
        correction.index = correction.index.astype(int)
        MONTH_CORRECTION_FACTORS = correction.astype(float)


def feature_importance(bundle: FittedBundle) -> pd.DataFrame:
    rows = []
    feature_names = bundle.revenue_cols

    lgb_importance = bundle.models["rev_lgb"].feature_importances_.astype(float)
    xgb_importance = bundle.models["rev_xgb"].feature_importances_.astype(float)
    ridge = bundle.models["rev_ridge"]
    ridge_importance = np.abs(ridge.named_steps["ridge"].coef_.astype(float))

    for source_name, values in [
        ("rev_lgb", lgb_importance),
        ("rev_xgb", xgb_importance),
        ("rev_ridge_abs_coef", ridge_importance),
    ]:
        total = values.sum()
        if total <= 0:
            normalized = values
        else:
            normalized = values / total
        for feature, value in zip(feature_names, normalized):
            rows.append({"model": source_name, "feature": feature, "importance": float(value)})

    importance = pd.DataFrame(rows)
    summary = (
        importance.groupby("feature", as_index=False)["importance"]
        .mean()
        .sort_values("importance", ascending=False)
    )
    return summary


def markdown_table(frame: pd.DataFrame, float_digits: int = 4) -> str:
    if frame.empty:
        return "_No rows._"
    formatted = frame.copy()
    for col in formatted.columns:
        if pd.api.types.is_float_dtype(formatted[col]):
            formatted[col] = formatted[col].map(lambda value: f"{value:.{float_digits}f}")
    headers = [str(col) for col in formatted.columns]
    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join(["---"] * len(headers)) + " |",
    ]
    for _, row in formatted.iterrows():
        lines.append("| " + " | ".join(str(row[col]) for col in formatted.columns) + " |")
    return "\n".join(lines)


def write_report(
    cv_results: pd.DataFrame,
    importance: pd.DataFrame,
    submission: pd.DataFrame,
) -> None:
    top_features = importance.head(15).copy()
    cv_main = cv_results[~cv_results["fold"].str.contains(":", regex=False)].copy()
    cv_final = cv_main[cv_main["train_end"].eq("MAE_tuned_month_corrected")].copy()
    if cv_final.empty:
        cv_final = cv_main
    avg = cv_final[["MAE", "RMSE", "R2"]].mean(numeric_only=True)

    report_lines = [
        "# Phần 3 - Mô hình dự báo doanh thu",
        "",
        "## 1. Mục tiêu và chia dữ liệu",
        "",
        "Mục tiêu là dự báo cột Revenue cho giai đoạn test 2023-01-01 đến 2024-07-01. Notebook đọc sales.csv và chia theo đúng PDF: train đến hết 2022-12-31. Với bộ local hiện tại tập test thật không được công bố, sample_submission.csv được dùng làm template để lấy ngày và giữ đúng thứ tự nộp.",
        "",
        "## 2. Kiểm soát leakage",
        "",
        "Pipeline dùng đầy đủ Revenue/COGS của tập train để tạo đặc trưng lag, rolling, EWM, margin và momentum. Phần bị cấm chỉ là Revenue/COGS thật của tập test. Vì vậy, khi dự báo test, notebook không đưa COGS trong template vào feature; COGS template chỉ được copy lại vào submission.csv sau khi đã dự báo Revenue để giữ đúng schema yêu cầu.",
        "",
        "## 3. Feature engineering",
        "",
        "Các nhóm đặc trưng chính gồm: lịch ngày/tháng/thứ, Fourier seasonality theo năm và tuần, lag 1-730 ngày của Revenue/COGS train, rolling mean/median/std/min/max của Revenue/COGS train, EWM, momentum, year-over-year ratio, biên lợi nhuận lịch sử và các tương tác giữa COGS train/current-predicted với lag/rolling/margin. Khi fit revenue model trên train, COGS cùng ngày của train được dùng hợp lệ vì đó là dữ liệu train. Khi dự báo test, COGS test không được dùng; notebook dự báo COGS nội sinh từ lịch sử train rồi mới dùng COGS dự báo này làm tín hiệu quy mô cho Revenue. Notebook cũng thêm revenue models không dùng current COGS, margin baselines và các model time-series chuyên dụng (ETS, ThetaModel, SARIMAX weekly) để giảm lỗi lan truyền từ bước dự báo COGS và tăng độ đa dạng của ensemble.",
        "",
        "## 4. Mô hình và tối ưu MAE",
        "",
        "Mô hình là ensemble của LightGBM, XGBoost, HistGradientBoosting, Ridge, seasonal baselines và các ứng viên time-series cổ điển tùy chọn. Boosting candidates gồm cả objective MAE/absolute-error và objective L2/squared-error: nhánh MAE giúp leaderboard, nhánh L2 giúp RMSE/R2. Notebook huấn luyện cả log-target models và raw-target models; log-target giúp ổn định với outlier, raw-target tối ưu trực tiếp trên thang đo doanh thu. Ensemble weights được tìm bằng OOF predictions theo objective đa metric: MAE là tiêu chí chính, sau đó chọn nghiệm có RMSE thấp hơn và R2 cao hơn trong nhóm nghiệm có MAE gần tốt nhất. Cách này bám leaderboard MAE nhưng không hy sinh chất lượng mô hình.",
        "",
        "Sau ensemble, notebook áp dụng correction theo tháng học từ fold gần nhất. Correction được shrink và clip rất bảo thủ để sửa bias mùa vụ còn sót lại mà không biến validation gần nhất thành leaderboard hack. Weight search cũng áp cap từng model để tránh một candidate đơn lẻ chiếm quá nhiều trọng số. ",
        "",
        "## 5. Cross-validation",
        "",
        markdown_table(cv_final, float_digits=4),
        "",
        f"Trung bình các dòng final: MAE={avg['MAE']:.2f}, RMSE={avg['RMSE']:.2f}, R2={avg['R2']:.4f}.",
        "",
        "## 6. Ensemble weights và correction",
        "",
        markdown_table(MAE_TUNED_WEIGHT_TABLE if not MAE_TUNED_WEIGHT_TABLE.empty else pd.DataFrame(), float_digits=6),
        "",
        "Month correction factors:",
        "",
        markdown_table(MONTH_CORRECTION_FACTORS.rename("factor").reset_index().rename(columns={"index": "month"}), float_digits=6) if not MONTH_CORRECTION_FACTORS.empty else "_Chưa có correction factors._",
        "",
        "",
        "## 7. Explainability",
        "",
        "Top feature importance tổng hợp từ LightGBM, XGBoost và trị tuyệt đối hệ số Ridge:",
        "",
        markdown_table(top_features, float_digits=6),
        "",
        "Diễn giải kinh doanh:",
        "",
        "- Lag 364/365/730 ngày và Fourier theo ngày trong năm cho thấy doanh thu có mùa vụ năm rõ rệt. Mô hình học được các giai đoạn cao điểm xuân-hè và vùng thấp điểm cuối năm.",
        "- Rolling mean/EWM 7-365 ngày thể hiện demand gần đây và xu hướng trung hạn là tín hiệu quan trọng cho dự báo ngày tiếp theo.",
        "- COGS dự báo nội sinh đại diện cho quy mô giá vốn/volume kỳ vọng, giúp mô hình học quan hệ biên lợi nhuận giữa Revenue và COGS mà không dùng COGS test.",
        "- Notebook cố ý không dùng các bảng transaction/master/operational khác trong model Phần 3 để tránh nguy cơ tái tạo trực tiếp Revenue/COGS. Những bảng đó phù hợp hơn cho EDA/Phần 2 hoặc phân tích giải thích bổ sung.",
        "",
        "## 8. Submission",
        "",
        f"Số dòng: {len(submission)}. Ngày đầu: {submission['Date'].iloc[0]}. Ngày cuối: {submission['Date'].iloc[-1]}.",
    ]
    (REPORT_DIR / "sales_forecasting_report.md").write_text("\n".join(report_lines), encoding="utf-8")


def load_data() -> tuple[pd.DataFrame, pd.DataFrame]:
    full_sales = pd.read_csv(DATA_DIR / "sales.csv", parse_dates=["Date"])
    full_sales = full_sales.sort_values("Date").reset_index(drop=True)

    train = full_sales[full_sales["Date"] <= TRAIN_END].copy()
    train = train.rename(columns={"Date": "date"}).reset_index(drop=True)

    sample_path = DATA_DIR / "sample_submission.csv"
    if sample_path.exists():
        test = pd.read_csv(sample_path, parse_dates=["Date"])
    else:
        test = full_sales[
            (full_sales["Date"] >= TEST_START) & (full_sales["Date"] <= TEST_END)
        ][["Date", "Revenue", "COGS"]].copy()

    return train, test


def build_submission(sales: pd.DataFrame, sample: pd.DataFrame) -> tuple[pd.DataFrame, FittedBundle]:
    predictions, bundle = forecast(sales, sample["Date"])
    revenue = predictions["revenue_pred"].copy()
    if APPLY_MONTH_CORRECTION and not MONTH_CORRECTION_FACTORS.empty:
        forecast_month = pd.to_datetime(predictions["date"]).dt.month
        revenue = revenue * forecast_month.map(MONTH_CORRECTION_FACTORS).fillna(1.0).to_numpy()

    submission = sample[["Date"]].copy()
    submission["Revenue"] = revenue.round(2)
    submission["COGS"] = predictions["cogs_pred"].round(2)
    submission["Date"] = submission["Date"].dt.strftime("%Y-%m-%d")
    return submission, bundle


## 2. Đọc dữ liệu và chia Train/Test

Theo PDF, Phần 3 dùng `sales.csv` làm train và `sales_test.csv` làm test. Trong bộ local hiện tại không có `sales_test.csv`, nên notebook dùng `sample_submission.csv` làm template ngày test. Đây chỉ là template để giữ schema/thứ tự dòng; model không dùng `Revenue` hoặc `COGS` của template làm feature.

Notebook ưu tiên đọc `sample_submission.csv` làm test template nếu file tồn tại. Nhờ vậy `submission.csv` sinh ra luôn giữ đúng số dòng và đúng thứ tự như template Kaggle, kể cả khi workspace có thêm file khác.


In [ ]:
sales, test = load_data()
full_sales_check = pd.read_csv(DATA_DIR / "sales.csv", parse_dates=["Date"])

print("Full sales.csv date range:", full_sales_check["Date"].min().date(), "->", full_sales_check["Date"].max().date())
print("Train shape:", sales.shape)
print("Train date range:", sales["date"].min().date(), "->", sales["date"].max().date())
print("Test/template shape:", test.shape)
print("Test/template date range:", test["Date"].min().date(), "->", test["Date"].max().date())
print("Columns:", list(test.columns))

assert sales["date"].max() <= TRAIN_END
assert list(test.columns) == ["Date", "Revenue", "COGS"]
assert test["Date"].is_monotonic_increasing
assert sales["date"].max() < test["Date"].min()


## 3. Cross-validation theo thời gian

Cell này chạy rolling-origin backtests với horizon 548 ngày. Nếu chỉ muốn tạo nhanh submission sau khi đã có kết quả CV và weight tuning, đặt `RUN_BACKTEST = False`. Khi chạy đầy đủ, notebook sẽ tạo OOF predictions, tìm ensemble weights theo MAE có trọng số thời gian, học correction theo tháng và lưu các artefact vào `04_report_data/`.

Cách đọc kết quả:

- Dòng `recent_2021H2_2022_month_corrected` là proxy gần nhất cho public test vì validation kết thúc đúng 2022-12-31.
- Trung bình ba fold dùng cho report để thể hiện độ ổn định.
- Nếu leaderboard MAE khác CV, ưu tiên điều chỉnh nhẹ correction/weights thay vì dùng bất kỳ thông tin test nào.


Notebook dùng cap trọng số từng candidate để tránh một model thắng CV cục bộ nhưng làm public score xấu. Đây là regularization cho ensemble, tương tự chống overfit ở model đơn.


In [ ]:
RUN_BACKTEST = True

REPORT_DIR.mkdir(parents=True, exist_ok=True)
if RUN_BACKTEST:
    cv_results = run_backtests(sales)
    cv_results.to_csv(REPORT_DIR / "cv_results.csv", index=False)
else:
    cv_results = pd.read_csv(REPORT_DIR / "cv_results.csv")
    load_existing_mae_weights()

cv_main = cv_results[~cv_results["fold"].str.contains(":", regex=False)].copy()
cv_final = cv_main[cv_main["train_end"].eq("MAE_tuned_month_corrected")].copy()
if cv_final.empty:
    raise RuntimeError("Month-corrected rows are missing. Re-run the CV cell after the latest notebook changes.")

recent_base = cv_main[cv_main["fold"].eq("recent_2021H2_2022")].iloc[0]
recent_final = cv_final[cv_final["fold"].eq("recent_2021H2_2022_month_corrected")].iloc[0]

display(cv_main)
print("Trọng số fold dùng để tune final ensemble đa metric:")
print(FINAL_TUNING_FOLD_WEIGHTS)
print("Recent fold trước hiệu chỉnh tháng (không phải bản final):")
print(recent_base[["MAE", "RMSE", "R2"]])
print("Metric proxy cho submission final = recent fold sau hiệu chỉnh tháng:")
print(recent_final[["MAE", "RMSE", "R2"]])
print("Mức cải thiện MAE từ correction:", float(recent_base["MAE"] - recent_final["MAE"]))
print("Trung bình các dòng final sau correction, chỉ để tham khảo report:")
print(cv_final[["MAE", "RMSE", "R2"]].mean(numeric_only=True))
print("Weight caps dùng để chống ensemble overfit:")
print(MODEL_WEIGHT_CAPS)
print("Ensemble weights đa metric dùng cho submission final:")
display(MAE_TUNED_WEIGHT_TABLE)
print("Hệ số hiệu chỉnh theo tháng dùng cho submission final:")
display(MONTH_CORRECTION_FACTORS.rename("factor").to_frame())


## 4. Huấn luyện trên toàn bộ train và tạo submission

Sau khi CV tìm được trọng số ensemble, notebook fit lại model trên toàn bộ giai đoạn train 2012-07-04 đến 2022-12-31, rồi forecast tuần tự cho đúng các ngày trong `sample_submission.csv`. File nộp giữ nguyên thứ tự dòng và cột `Date,Revenue,COGS`. Cả `Revenue` và `COGS` trong submission đều là output dự báo của model; template chỉ dùng để lấy lịch ngày và format.


In [ ]:
# Submission final dùng ensemble weights đa metric và correction theo tháng có kiểm soát.
if APPLY_MONTH_CORRECTION and MONTH_CORRECTION_FACTORS.empty:
    correction_path = REPORT_DIR / "month_correction_factors.csv"
    if correction_path.exists():
        correction = pd.read_csv(correction_path, index_col=0).iloc[:, 0]
        correction.index = correction.index.astype(int)
        MONTH_CORRECTION_FACTORS = correction.astype(float)
    else:
        raise RuntimeError("No month correction factors found. Run the CV cell first.")

submission, bundle = build_submission(sales, test)
importance = feature_importance(bundle)

importance.to_csv(REPORT_DIR / "feature_importance.csv", index=False)
submission.to_csv(ROOT / "submission.csv", index=False)
submission.to_csv(REPORT_DIR / "submission.csv", index=False)
write_report(cv_results, importance, submission)

print("Đã lưu:", ROOT / "submission.csv")
print("Số dòng:", len(submission))
print("Khoảng ngày:", submission["Date"].iloc[0], "->", submission["Date"].iloc[-1])
print("Đã áp dụng month correction:", bool(APPLY_MONTH_CORRECTION and not MONTH_CORRECTION_FACTORS.empty))
print("COGS trong submission là dự báo nội sinh, không copy từ template/test.")
display(MONTH_CORRECTION_FACTORS.rename("factor").to_frame())
display(submission.head())
display(submission.tail())


## 5. Giải thích mô hình

Phần này tổng hợp feature importance từ LightGBM, XGBoost và Ridge. Cách đọc theo ngôn ngữ kinh doanh: feature mùa vụ cho biết thời điểm trong năm/tuần có ảnh hưởng mạnh; lag/rolling cho biết demand gần đây và cùng kỳ năm trước; margin và COGS dự báo cho biết quy mô giá vốn/volume kỳ vọng đang kéo doanh thu lên hay xuống.


In [ ]:
display(importance.head(20))

print("Diễn giải kinh doanh:")
print("- Các lag theo năm, Fourier terms và month dummies cho thấy doanh thu có mùa vụ mạnh theo lịch năm.")
print("- Rolling/EWM/momentum features thể hiện demand gần đây, quán tính và thay đổi xu hướng trung hạn.")
print("- Revenue/COGS của TRAIN là lõi feature engineering: lag, rolling, EWM, momentum, margin và tương tác COGS-margin.")
print("- COGS dự báo nội sinh đóng vai trò tín hiệu quy mô/volume ở test, không dùng COGS test.")
print("- Ensemble có thêm seasonal blend, same-day-of-year window, local trend và ETS/Theta/SARIMAX tùy chọn, tất cả fit từ train history.")
print("- COGS trong submission là forecast nội sinh; thứ tự Date vẫn giữ y hệt sample_submission.csv.")


## 6. Kiểm tra định dạng file nộp

Cell này xác nhận submission có đúng cột, đúng số dòng, đúng thứ tự ngày như `sample_submission.csv`, và không có giá trị âm/missing. Không kiểm tra COGS bằng template nữa vì COGS trong file nộp là dự báo nội sinh của model, không phải dữ liệu test được dùng lại.


In [ ]:
template_check = pd.read_csv(DATA_DIR / "sample_submission.csv", parse_dates=["Date"])
template_check["Date"] = template_check["Date"].dt.strftime("%Y-%m-%d")

sub_check = pd.read_csv(ROOT / "submission.csv")

checks = {
    "same_columns": list(sub_check.columns) == ["Date", "Revenue", "COGS"],
    "same_rows": len(sub_check) == len(template_check),
    "same_date_order": sub_check["Date"].equals(template_check["Date"]),
    "revenue_missing": int(sub_check["Revenue"].isna().sum()),
    "revenue_nonpositive": int((sub_check["Revenue"] <= 0).sum()),
    "cogs_missing": int(sub_check["COGS"].isna().sum()),
    "cogs_nonpositive": int((sub_check["COGS"] <= 0).sum()),
}
print(checks)
assert all(value is True or value == 0 for value in checks.values())
sub_check.describe()
